# Tile Download Verification. 

How many tiles were downloaded per product
Missing tiles (expected from CSVs but not on disk)
Tile integrity: dimensions (534x534), file size, band count
Visual inspection of sample tiles

In [ ]:
import os
import csv
import json
import glob
from pathlib import Path
from collections import defaultdict

import rasterio
import numpy as np
import matplotlib.pyplot as plt

# ── Configuration ──
# Adjust these paths for cluster vs local
SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))

# Auto-detect environment
if os.path.exists(os.path.expanduser("~/thesis_tiles")):
    TILES_DIR = os.path.expanduser("~/thesis_tiles")
    CSV_DIR = os.path.join(SCRIPT_DIR, "data_csv")
else:
    TILES_DIR = "/Users/angelicamariamorenorojas/Desktop/Master/thesis/tiles"
    CSV_DIR = os.path.join(SCRIPT_DIR, "data_csv")

PRODUCTS = ["s2_l2a", "s2_l1c", "s1_grd"]
WINDOWS = ["evt", "bef", "aft"]

EXPECTED_BANDS = {
    "s2_l2a": 12,
    "s2_l1c": 13,
    "s1_grd": 2,
}

EXPECTED_SIZE = 534  # pixels

print(f"Tiles directory: {TILES_DIR}")
print(f"CSV directory:   {CSV_DIR}")
print(f"Products:        {PRODUCTS}")

## 1. Scan downloaded tiles

In [ ]:
# Count all .tif files per product and window
downloaded = defaultdict(lambda: defaultdict(list))  # product -> window -> [paths]

for product in PRODUCTS:
    product_dir = os.path.join(TILES_DIR, product)
    if not os.path.exists(product_dir):
        print(f"WARNING: {product_dir} does not exist")
        continue
    for tif in glob.glob(os.path.join(product_dir, "fid_*", "*", "*", "*.tif")):
        # path: .../product/fid_X/window/image_id/tile_N.tif
        parts = Path(tif).parts
        # Find window from path
        fid_idx = next(i for i, p in enumerate(parts) if p.startswith("fid_"))
        window = parts[fid_idx + 1]
        downloaded[product][window].append(tif)

# Summary table
print(f"{'Product':<12} {'evt':>8} {'bef':>8} {'aft':>8} {'TOTAL':>8}")
print("-" * 48)
grand_total = 0
for product in PRODUCTS:
    counts = {w: len(downloaded[product][w]) for w in WINDOWS}
    total = sum(counts.values())
    grand_total += total
    print(f"{product:<12} {counts['evt']:>8} {counts['bef']:>8} {counts['aft']:>8} {total:>8}")
print("-" * 48)
print(f"{'TOTAL':<12} {'':>8} {'':>8} {'':>8} {grand_total:>8}")

## 2. Expected vs Downloaded (from CSVs)

In [ ]:
# Parse CSVs and build the expected set of (fid, sensor, window, image_id)
IMAGE_ID_COLUMNS = {
    ("s2", "evt"): "evtIdsS2", ("s2", "bef"): "befIdsS2", ("s2", "aft"): "aftIdsS2",
    ("s1", "evt"): "evtIdsS1", ("s1", "bef"): "befIdsS1", ("s1", "aft"): "aftIdsS1",
}

CSV_FILES = ["v1_images_s2_s1.csv", "v2_images_s2_s1.csv", "v3_images_s2_s1.csv"]

# fid -> {(sensor, window): set of image_ids}
expected = defaultdict(lambda: defaultdict(set))

for csv_name in CSV_FILES:
    csv_path = os.path.join(CSV_DIR, csv_name)
    if not os.path.exists(csv_path):
        print(f"WARNING: {csv_path} not found")
        continue
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            fid = row["fid"].strip()
            for (sensor, window), col in IMAGE_ID_COLUMNS.items():
                raw = row.get(col, "").strip()
                if raw:
                    ids = [x.strip() for x in raw.split(",") if x.strip()]
                    expected[fid][(sensor, window)].update(ids)

# Count expected unique downloads per product
expected_counts = defaultdict(lambda: defaultdict(int))
for fid, sw_dict in expected.items():
    for (sensor, window), ids in sw_dict.items():
        if sensor == "s2":
            for p in ["s2_l2a", "s2_l1c"]:
                expected_counts[p][window] += len(ids)
        else:
            expected_counts["s1_grd"][window] += len(ids)

print(f"Total FIDs in CSVs: {len(expected)}")
print()
print(f"{'Product':<12} {'evt':>8} {'bef':>8} {'aft':>8} {'TOTAL':>8}")
print("-" * 48)
for product in PRODUCTS:
    counts = {w: expected_counts[product][w] for w in WINDOWS}
    total = sum(counts.values())
    print(f"{product:<12} {counts['evt']:>8} {counts['bef']:>8} {counts['aft']:>8} {total:>8}")

In [ ]:
# Find which FIDs were actually downloaded (from directory names)
downloaded_fids = set()
for product in PRODUCTS:
    product_dir = os.path.join(TILES_DIR, product)
    if not os.path.exists(product_dir):
        continue
    for d in os.listdir(product_dir):
        if d.startswith("fid_"):
            downloaded_fids.add(d.replace("fid_", ""))

expected_fids = set(expected.keys())
missing_fids = expected_fids - downloaded_fids
extra_fids = downloaded_fids - expected_fids

print(f"FIDs expected (from CSVs): {len(expected_fids)}")
print(f"FIDs downloaded:           {len(downloaded_fids)}")
print(f"FIDs missing:              {len(missing_fids)}")
print(f"FIDs extra (unexpected):   {len(extra_fids)}")

if missing_fids and len(missing_fids) <= 20:
    print(f"\nMissing FIDs: {sorted(missing_fids)}")

## 3. Tile integrity check (dimensions, bands, file size)

In [ ]:
# Check a sample of tiles per product for dimensions, bands, and corruption
import random
random.seed(42)

MAX_CHECK = 50  # tiles to check per product

issues = []

for product in PRODUCTS:
    all_tiles = []
    for w in WINDOWS:
        all_tiles.extend(downloaded[product][w])
    
    if not all_tiles:
        print(f"{product}: no tiles found, skipping")
        continue
    
    sample = random.sample(all_tiles, min(MAX_CHECK, len(all_tiles)))
    
    ok = 0
    bad_size = 0
    bad_bands = 0
    corrupted = 0
    zero_size = 0
    sizes_mb = []
    
    for tif_path in sample:
        fsize = os.path.getsize(tif_path)
        sizes_mb.append(fsize / 1e6)
        
        if fsize == 0:
            zero_size += 1
            issues.append((product, tif_path, "empty file (0 bytes)"))
            continue
        
        try:
            with rasterio.open(tif_path) as src:
                h, w = src.height, src.width
                bands = src.count
                
                if h != EXPECTED_SIZE or w != EXPECTED_SIZE:
                    bad_size += 1
                    issues.append((product, tif_path, f"size {w}x{h} (expected {EXPECTED_SIZE}x{EXPECTED_SIZE})"))
                elif bands != EXPECTED_BANDS[product]:
                    bad_bands += 1
                    issues.append((product, tif_path, f"{bands} bands (expected {EXPECTED_BANDS[product]})"))
                else:
                    ok += 1
        except Exception as e:
            corrupted += 1
            issues.append((product, tif_path, f"corrupted: {e}"))
    
    print(f"\n{product} (checked {len(sample)} / {len(all_tiles)} tiles):")
    print(f"  OK:        {ok}")
    print(f"  Bad size:  {bad_size}")
    print(f"  Bad bands: {bad_bands}")
    print(f"  Corrupted: {corrupted}")
    print(f"  Empty:     {zero_size}")
    print(f"  File size: {np.mean(sizes_mb):.1f} MB avg, {np.min(sizes_mb):.1f}-{np.max(sizes_mb):.1f} MB range")

In [ ]:
# Print all issues found
if issues:
    print(f"ISSUES FOUND: {len(issues)}\n")
    for product, path, msg in issues[:30]:
        short_path = "/".join(Path(path).parts[-5:])
        print(f"  [{product}] {short_path}: {msg}")
    if len(issues) > 30:
        print(f"  ... and {len(issues) - 30} more")
else:
    print("No issues found - all checked tiles are valid!")

## 4. Missing images per FID (downloaded FIDs only)

In [ ]:
# For each downloaded FID, check which image_ids are on disk vs expected
# Build a set of downloaded (product, fid, window, image_id) from file paths
on_disk = set()
for product in PRODUCTS:
    for window in WINDOWS:
        for tif_path in downloaded[product][window]:
            parts = Path(tif_path).parts
            fid_idx = next(i for i, p in enumerate(parts) if p.startswith("fid_"))
            fid = parts[fid_idx].replace("fid_", "")
            image_id = parts[fid_idx + 2]  # window is +1, image_id is +2
            on_disk.add((product, fid, window, image_id))

# Build expected set for downloaded FIDs only
SENSOR_TO_PRODUCTS = {"s2": ["s2_l2a", "s2_l1c"], "s1": ["s1_grd"]}

missing_per_product = defaultdict(int)
total_expected_for_downloaded = defaultdict(int)

for fid in downloaded_fids:
    if fid not in expected:
        continue
    for (sensor, window), ids in expected[fid].items():
        for product in SENSOR_TO_PRODUCTS.get(sensor, []):
            for img_id in ids:
                safe_id = img_id.replace("/", "_")
                total_expected_for_downloaded[product] += 1
                if (product, fid, window, safe_id) not in on_disk:
                    missing_per_product[product] += 1

print("For downloaded FIDs only:")
print(f"{'Product':<12} {'Expected':>10} {'Missing':>10} {'Complete %':>12}")
print("-" * 46)
for product in PRODUCTS:
    exp = total_expected_for_downloaded[product]
    mis = missing_per_product[product]
    pct = ((exp - mis) / exp * 100) if exp > 0 else 0
    print(f"{product:<12} {exp:>10} {mis:>10} {pct:>11.1f}%")

## 5. Visual inspection (random samples)

In [ ]:
import geopandas as gpd
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
from rasterio.transform import from_bounds

# Load shapefile with polygons
SHP_DIR = os.path.join(SCRIPT_DIR, "data_shp")
if os.path.exists(os.path.expanduser("~/thesis_scripts/data_shp")):
    SHP_DIR = os.path.expanduser("~/thesis_scripts/data_shp")

shp_path = os.path.join(SHP_DIR, "label_polygons.shp")
gdf = gpd.read_file(shp_path)

# Ensure fid column is string for matching
if "fid" in gdf.columns:
    gdf["fid"] = gdf["fid"].astype(str)
else:
    # Try system:index or other ID columns
    for col in ["system:index", "id", "FID"]:
        if col in gdf.columns:
            gdf["fid"] = gdf[col].astype(str)
            break

print(f"Loaded {len(gdf)} polygons from {shp_path}")
print(f"CRS: {gdf.crs}")


def overlay_polygon(ax, tif_path, fid, gdf):
    """Overlay the deforestation polygon on a tile plot."""
    poly_row = gdf[gdf["fid"] == str(fid)]
    if poly_row.empty:
        return

    with rasterio.open(tif_path) as src:
        tile_crs = src.crs
        tile_transform = src.transform
        tile_h, tile_w = src.height, src.width

    # Reproject polygon to tile CRS
    poly_reproj = poly_row.to_crs(tile_crs)

    # Convert polygon coordinates to pixel coordinates
    for geom in poly_reproj.geometry:
        if geom.is_empty:
            continue
        if geom.geom_type == "Polygon":
            geoms = [geom]
        elif geom.geom_type == "MultiPolygon":
            geoms = list(geom.geoms)
        else:
            continue

        for g in geoms:
            xs, ys = g.exterior.coords.xy
            # Transform from CRS coords to pixel coords
            pixel_coords = []
            for x, y in zip(xs, ys):
                col, row = ~tile_transform * (x, y)
                pixel_coords.append((col, row))

            poly_patch = MplPolygon(
                pixel_coords, closed=True,
                edgecolor="yellow", facecolor="none", linewidth=2
            )
            ax.add_patch(poly_patch)


def plot_s2_rgb(tif_path, ax, title="", fid=None, gdf=None):
    """Plot S2 tile as RGB (B4, B3, B2) with optional polygon overlay."""
    with rasterio.open(tif_path) as src:
        r = src.read(4).astype(float)  # B4 - Red
        g = src.read(3).astype(float)  # B3 - Green
        b = src.read(2).astype(float)  # B2 - Blue

    rgb = np.stack([r, g, b], axis=-1)
    rgb = np.clip(rgb / 3000, 0, 1)
    ax.imshow(rgb)
    if fid is not None and gdf is not None:
        overlay_polygon(ax, tif_path, fid, gdf)
    ax.set_title(title, fontsize=8)
    ax.axis("off")


def plot_s1_vv(tif_path, ax, title="", fid=None, gdf=None):
    """Plot S1 tile VV band with optional polygon overlay."""
    with rasterio.open(tif_path) as src:
        vv = src.read(1).astype(float)
    ax.imshow(vv, cmap="gray", vmin=-25, vmax=0)
    if fid is not None and gdf is not None:
        overlay_polygon(ax, tif_path, fid, gdf)
    ax.set_title(title, fontsize=8)
    ax.axis("off")


# Pick a random FID that has event tiles in all products
sample_fid = None
for fid in random.sample(sorted(downloaded_fids), min(20, len(downloaded_fids))):
    has_all = True
    for product in PRODUCTS:
        product_evt = os.path.join(TILES_DIR, product, f"fid_{fid}", "evt")
        if not os.path.exists(product_evt) or not os.listdir(product_evt):
            has_all = False
            break
    if has_all:
        sample_fid = fid
        break

if sample_fid:
    print(f"Showing tiles for FID {sample_fid}")

    fig, axes = plt.subplots(len(PRODUCTS), 3, figsize=(12, 4 * len(PRODUCTS)))
    if len(PRODUCTS) == 1:
        axes = axes[np.newaxis, :]

    for row, product in enumerate(PRODUCTS):
        for col, window in enumerate(WINDOWS):
            ax = axes[row, col]
            window_dir = os.path.join(TILES_DIR, product, f"fid_{sample_fid}", window)

            if not os.path.exists(window_dir):
                ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
                ax.set_title(f"{product} / {window}", fontsize=8)
                ax.axis("off")
                continue

            tifs = glob.glob(os.path.join(window_dir, "*", "tile_0.tif"))
            if not tifs:
                ax.text(0.5, 0.5, "no tiles", ha="center", va="center", transform=ax.transAxes)
                ax.set_title(f"{product} / {window}", fontsize=8)
                ax.axis("off")
                continue

            tif = tifs[0]
            img_id = Path(tif).parent.name[:30]
            title = f"{product} / {window}\n{img_id}..."

            if product.startswith("s2"):
                plot_s2_rgb(tif, ax, title, fid=sample_fid, gdf=gdf)
            else:
                plot_s1_vv(tif, ax, title, fid=sample_fid, gdf=gdf)

    plt.suptitle(f"FID {sample_fid} — yellow = deforestation polygon", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No FID found with event tiles in all products")

## 6. Disk usage summary

In [ ]:
# Disk usage per product
def dir_size_gb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / 1e9

print(f"{'Product':<12} {'Size (GB)':>10} {'Files':>8}")
print("-" * 32)
total_gb = 0
total_files = 0
for product in PRODUCTS:
    product_dir = os.path.join(TILES_DIR, product)
    if os.path.exists(product_dir):
        gb = dir_size_gb(product_dir)
        n_files = sum(len(downloaded[product][w]) for w in WINDOWS)
        total_gb += gb
        total_files += n_files
        print(f"{product:<12} {gb:>10.2f} {n_files:>8}")
    else:
        print(f"{product:<12} {'N/A':>10} {'N/A':>8}")
print("-" * 32)
print(f"{'TOTAL':<12} {total_gb:>10.2f} {total_files:>8}")